In [ ]:
import os
import math
import json
import numpy as np
import random

# 런팟
os.environ["WANDB_PROJECT"] = "patent_disc"
os.environ["HF_HOME"] = "/workspace/hf_cache"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from datasets import load_dataset, load_from_disk
from dataclasses import dataclass
from sklearn.metrics import f1_score

In [ ]:
# Config
SEARCH = False   # True=레시피 탐색(짧은 런) / False=최종 풀런

config = {
    "num_labels": 188,
    "seed": 42,
    "learning_rate": 3e-5,
    "epochs": 2 if SEARCH else 12,
    "early_stop": 6,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "model_name": "skt/A.X-Encoder-base",
    "max_len": 512,          
    "eff_batch": 128,           
    "micro_batch": 128,
    "eval_micro_batch": 512,
    "gamma_pos": 0,             # ASL 양성 포커싱
    "gamma_neg": 4,            # ASL 음성 포커싱
    "margin": 0.05,            # ASL 확률 시프팅 m
    "repo_final": "ingyoun/A.X-patent-len512-ASL",
    "out_path": "/workspace/output/modernbert-len512_ASL",
    "tag": "modernbert-patent-len512-ASL",
    "rev": "9708f9c404ace91efd25c06fac2d73413616f4ef",
}

config["grad_accum"] = config["eff_batch"] // config["micro_batch"]   # micro×accum = eff_batch
config["run_name"] = (
    f"axenc_len512_ASL"
    + ("_search" if SEARCH else "_full")
)

# wandb는 코드 캡처를 위해 실제 노트북 파일 경로를 요구한다(bare 파일명이면 경고). 실행 위치 기준 절대경로로 지정
os.environ["WANDB_NOTEBOOK_NAME"] = os.path.abspath("09_02_Loss_ASL.ipynb")

In [ ]:
random.seed(config['seed'])
np.random.seed(config['seed'])
torch.manual_seed(config['seed'])
torch.cuda.manual_seed_all(config['seed'])

In [ ]:
print(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

## 데이터셋

In [ ]:
dataset = load_dataset(
    "ingyoun/patent-clean-text-modernbert-tokenized",
    cache_dir="/workspace/hf_cache",
)

dataset

## 모델

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
        pretrained_model_name_or_path=config["model_name"],
        num_labels=config["num_labels"], 
        problem_type="multi_label_classification", 
        classifier_dropout=0.5,
        dtype=torch.float32,
        attn_implementation="flash_attention_2",            # len8192와 구현 일치
    )

## 토크나이저

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(config["model_name"], revision=config["rev"])
EOS_ID = tokenizer.eos_token_id


def _prep(batch):
    max_len = config["max_len"]
    ids, masks = [], []
    for x, m in zip(batch["input_ids"], batch["attention_mask"]):
        if len(x) > max_len:
            x = x[: max_len - 1] + [EOS_ID]   # <s> 유지 + 꼬리를 <\s>로 마감
            m = m[:max_len]
        ids.append(x)
        masks.append(m)
    return {"input_ids": ids, "attention_mask": masks, "length": [len(i) for i in ids]}


# 절단은 max_len에만 의존 → 길이별로 볼륨에 캐시. 재훈련 시 절단(.map) 생략, max_len 변경 시 새 경로로 재생성
prep_cache = f"/workspace/prep_cache/len{config['max_len']}"
if os.path.isdir(prep_cache):
    dataset = load_from_disk(prep_cache)
    print(f"prep 캐시 로드: {prep_cache}")
else:
    dataset = dataset.map(_prep, batched=True)
    dataset.save_to_disk(prep_cache)
    print(f"prep 캐시 저장: {prep_cache}")

print(f"EOS_ID={EOS_ID}  max_len={config['max_len']}")
dataset

## 커스텀

$$L_{ASL} = \begin{cases} -(1-p)^{\gamma_+} \log p & (y = 1) \\ -(p_m)^{\gamma_-} \log(1 - p_m) & (y = 0) \end{cases}$$

$$p_m = \max(p - m, 0)$$

In [ ]:
class AsymmetricLoss(nn.Module):
    """Multi-label ASL (Ridnik et al., ICCV 2021)."""
    def __init__(self, gamma_pos: int = 0, gamma_neg: int = 4, margin: float = 0.05):
        super().__init__()
        self.gamma_pos = gamma_pos
        self.gamma_neg = gamma_neg
        self.margin = margin

    def forward(self, logits, labels):
        p = torch.sigmoid(logits)
        pm = (p - self.margin).clamp(min=0.0)       # p_m = max(p - m, 0)
        eps = 1e-8

        # 양성: (1-p)^γ+ · (-log p),  음성: (p_m)^γ- · (-log(1 - p_m))
        loss_pos = (1 - p) ** self.gamma_pos * torch.log(p.clamp(min=eps))
        loss_neg = pm ** self.gamma_neg * torch.log((1 - pm).clamp(min=eps))

        loss = labels * loss_pos + (1 - labels) * loss_neg
        return -loss.sum(dim=1).mean()

In [ ]:
@dataclass
class ASLTrainingArguments(TrainingArguments):
    gamma_pos: int = 0
    gamma_neg: int = 4
    margin: float = 0.05


class ASLTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_fn = AsymmetricLoss(
            gamma_pos=self.args.gamma_pos,
            gamma_neg=self.args.gamma_neg,
            margin=self.args.margin,
        )

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = self.loss_fn(outputs.logits, labels.float())
        return (loss, outputs) if return_outputs else loss

In [ ]:
class MultiLabelCollator:
    def __init__(self, tokenizer):
        self.tok = tokenizer
    
    def __call__(self, feats):
        labels = torch.tensor([f["labels"] for f in feats], dtype=torch.float)
        keys = ("input_ids", "attention_mask")
        enc = [{k: f[k] for k in keys if k in f} for f in feats]
        batch = self.tok.pad(enc, padding=True, return_tensors="pt")
        batch["labels"] = labels
        return batch

In [ ]:
def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def compute_metrics(eval_pred):
    """
    sigmoid→τ=0.5 멀티라벨 F1(micro/macro/sample).
    micro_f1이 모델 선택(metric_for_best_model) 기준. 벤더 연속성 앵커(top-1 weighted)도 병기.
    """
    logits, labels = eval_pred
    logits = np.asarray(logits)
    Y = np.asarray(labels).astype(int)
    pred = (_sigmoid(logits) >= 0.5).astype(int)
    return {
        "micro_f1":  f1_score(Y, pred, average="micro",   zero_division=0),   # headline (모델 선택 기준)
        "macro_f1":  f1_score(Y, pred, average="macro",   zero_division=0),
        "sample_f1": f1_score(Y, pred, average="samples", zero_division=0),
        "empty_rate": float((pred.sum(1) == 0).mean()),
        "anchor_weighted_f1": f1_score(Y.argmax(1), logits.argmax(1), average="weighted", zero_division=0),  # 벤더 연속성
    }

## 훈련

In [ ]:
steps_per_epoch = math.ceil(len(dataset["train"]) / config["eff_batch"])
num = 4 if SEARCH else 2
eval_steps = math.ceil(steps_per_epoch / num)   # 탐색:에폭당 4회 / 풀런:에폭당 2회

training_args = ASLTrainingArguments(
    output_dir='/workspace/results',
    seed=config["seed"],
    learning_rate=config["learning_rate"],
    weight_decay=config["weight_decay"],
    lr_scheduler_type="linear",
    warmup_ratio=config["warmup_ratio"],
    per_device_train_batch_size=config["micro_batch"],
    per_device_eval_batch_size=config["eval_micro_batch"],
    gradient_accumulation_steps=config["grad_accum"],   
    train_sampling_strategy="group_by_length",          # 유사 길이 배치로 padding 최소화
    remove_unused_columns=False,                        # 커스텀 collator가 키를 직접 선택 + length 컬럼 보존
    num_train_epochs=config["epochs"],
    bf16=True,                                          # ModernBERT 계열 안정성엔 bf16 (bf16=True, fp16=False)
    eval_strategy='steps',
    eval_steps=eval_steps,
    save_strategy="no" if SEARCH else "steps",          # 탐색:저장 안 함 / 풀런:에폭당 저장(볼륨)
    save_steps=eval_steps,                              # save_strategy="no"면 무시됨
    save_total_limit=6,
    logging_steps=50,
    metric_for_best_model="micro_f1",         
    greater_is_better=True,
    load_best_model_at_end=not SEARCH,                  # 풀런에서만 best 복원
    report_to="wandb",
    run_name=config["run_name"],
    gamma_pos=config["gamma_pos"],                      # ASL 파라미터를 args로 전달 → ASLTrainer가 self.args에서 읽음
    gamma_neg=config["gamma_neg"],
    margin=config["margin"],
)

In [ ]:
trainer = ASLTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    data_collator=MultiLabelCollator(tokenizer),
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=config["early_stop"])]
)

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()
print(f"잔여 allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB (모델 가중치)")

In [ ]:
trainer.train()

## 평가

In [ ]:
if not SEARCH:
    test_metrics = trainer.evaluate(dataset["test"], metric_key_prefix="test")
    for k, v in test_metrics.items():
        print(f"{k}: {v}")

In [ ]:
if not SEARCH:
    os.makedirs(config["out_path"], exist_ok=True)

    metrics_fp = os.path.join(config["out_path"], f"{config['tag']}_test_metrics.json")
    with open(metrics_fp, "w", encoding="utf-8") as f:
        json.dump(test_metrics, f, ensure_ascii=False, indent=2)

    print("saved", config["out_path"])

In [ ]:
if not SEARCH:
    print(trainer.state.best_model_checkpoint)
    print(trainer.state.best_metric)

In [ ]:
if not SEARCH:
    trainer.model.push_to_hub(config["repo_final"])
    tokenizer.push_to_hub(config["repo_final"])